# Error analysis — physics solver results

Reads a `results/*.csv` (or its JSON sibling), categorises every wrong row into a fail mode, and lets you eyeball examples per mode + per domain.

Edit `RESULTS_CSV` below to switch versions. The same notebook works for v01, v02, v03, ...

In [3]:
import os, sys
# When run from inside the repo, make the package importable.
if 'app' not in os.listdir('.'):
    # Walk up until we find it (handles being run from .ipynb's own dir).
    here = os.path.abspath('.')
    while here != os.path.dirname(here) and not os.path.isdir(os.path.join(here, 'app')):
        here = os.path.dirname(here)
    os.chdir(here)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

RESULTS_CSV = 'app/physics_solution/results/v02_fewshot.csv'  # change me
REPORT_OUT  = 'docs/eda/v02_error_report.md'                  # markdown to write
N_EXAMPLES  = 5

cwd: d:\Git\Exact_2026_Laplace-s_Red_Devils


In [4]:
from app.physics_solution.eda.error_analysis import analyse
report = analyse(RESULTS_CSV, n_examples_per_mode=N_EXAMPLES)
print(f'Total: {report.total}, correct: {report.correct} ({report.accuracy:.3f}), wrong: {report.wrong}')

OverflowError: cannot convert float infinity to integer

## 1. Fail-mode breakdown


In [ ]:
import pandas as pd
wrong = max(report.wrong, 1)
fm = (
    pd.Series(report.failmode_counts, name='count')
      .sort_values(ascending=False)
      .to_frame()
)
fm['% of wrong'] = (fm['count'] / wrong * 100).round(1)
fm

In [ ]:
# Bar chart of fail modes
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
fm['count'].plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel('# wrong rows')
ax.set_title(f'Fail modes — {report.version}')
plt.tight_layout()
plt.show()

## 2. Per-domain accuracy


In [ ]:
report.domain_breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
d = report.domain_breakdown.sort_values('accuracy')
ax.barh(d['domain'], d['accuracy'])
ax.set_xlim(0, 1)
ax.set_xlabel('accuracy')
ax.set_title(f'Per-domain accuracy — {report.version}')
for i, (acc, n) in enumerate(zip(d['accuracy'], d['n'])):
    ax.text(acc + 0.01, i, f'  n={n}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Fail mode × domain


In [ ]:
report.domain_x_failmode

In [ ]:
# Heatmap — quickly see which domains drive which fail modes.
if not report.domain_x_failmode.empty:
    fig, ax = plt.subplots(figsize=(10, 0.4 * len(report.domain_x_failmode) + 2))
    import numpy as np
    m = report.domain_x_failmode.values
    im = ax.imshow(m, cmap='Reds', aspect='auto')
    ax.set_xticks(range(report.domain_x_failmode.shape[1]))
    ax.set_xticklabels(report.domain_x_failmode.columns)
    ax.set_yticks(range(report.domain_x_failmode.shape[0]))
    ax.set_yticklabels(report.domain_x_failmode.index)
    for (i, j), val in np.ndenumerate(m):
        ax.text(j, i, int(val), ha='center', va='center',
                color='white' if val > m.max() * 0.5 else 'black', fontsize=9)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

## 4. Eyeball examples per fail mode

These should make the categorisation tangible — if a fail mode looks miscategorised, fix the heuristic in `error_analysis.py`.

In [ ]:
for mode, examples in report.examples_per_mode.items():
    print(f'=== {mode} ({report.failmode_counts.get(mode, 0)}) ===')
    for ex in examples:
        print(f"  {ex['id']:>10}  gold={ex['gold_answer']!s:>10}  pred={ex['pred_numeric']!s:>12}  "
              f"unit_gold={ex['gold_unit']!s:>6}  unit_pred={ex['pred_unit']!s:>6}")
        q = (ex.get('question') or '')[:180]
        print(f"             Q: {q}")
    print()

## 5. Drill into a single fail mode

Set `FOCUS_MODE` below to inspect more rows from one category — useful when designing v03/v04 fixes.

In [ ]:
FOCUS_MODE = 'magnitude_wrong'  # or 'off_by_power10' / 'unit_mismatch' / ...
subset = report.wrong_df[report.wrong_df['fail_mode'] == FOCUS_MODE]
print(f'Rows: {len(subset)}')
subset[['id', 'domain', 'gold_answer', 'pred_numeric', 'gold_unit', 'pred_unit']].head(20)

In [ ]:
# Full completion for one focus-mode example.
if not subset.empty:
    row = subset.iloc[0]
    print('ID:', row['id'])
    print('Question:', row['question'])
    print('Gold:', row['gold_answer'], row['gold_unit'])
    print('Pred:', row['pred_numeric'], row['pred_unit'])
    print()
    print('--- completion ---')
    print(str(row['completion'])[:2000])

## 6. Write the report


In [ ]:
out = report.write(REPORT_OUT)
print(f'Wrote {out}')